# 03 — Embeddings (from saved weights; reused when inputs are unchanged)

Each embedding file holds `idx, seq_key, domain, nfl_id, kind, position_group, fold, split_role, e0…` and its dependency hash (bundle, science config, sha256 of the weights). Changed inputs → a new file, never a silent reuse.

In [ ]:
%pip install -q "git+https://github.com/mybenkhadda/NFL_bowl.git"
from google.colab import drive
drive.mount("/content/drive")
from nflbowl import Experiment

exp = Experiment.open("/content/drive/MyDrive/NFLBowl/experiments/f4_main", data_dir="/content/drive/MyDrive/NFLBowl/bundles/m12f_v1",
                      runtime={"stage_dir": "/content/nflbowl_stage"})
exp.prepare()

In [ ]:
# baselines + every trained transductive encoder
for name in ["F0", "random"] + exp.list_models().filter(exp.list_models()["trained"] & (exp.list_models()["kind"] == "encoder"))["name"].to_list():
    try:
        E = exp.embed(name, reuse=True)
        print(name, E.shape)
    except FileNotFoundError as e:          # e.g. only the inductive runs exist
        print(name, "—", e)

In [ ]:
# inductive fold embeddings (one encoder per held-out fold)
by_fold = exp.embed("F4", regime="inductive", reuse=True)
{f: E.shape for f, E in by_fold.items()}

In [ ]:
# load a saved table without any computation (works without a GPU)
df = exp.load_embeddings("F4")
df.group_by("domain", "split_role").len()